In [ ]:
# 필요한 라이브러리 설치하기
!pip install langchain langchain-openai pypdf faiss-cpu tiktoken sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.4/55.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 415.4/415.4 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12

In [ ]:
!pip install -U langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 5.0 MB/s eta 0:00:00


In [ ]:
# 필요한 라이브러리 가져오기 및 API 키 설정
import os
from getpass import getpass
import tempfile
from google.colab import files

# API 키를 안전하게 입력받기
api_key = getpass("OpenAI API 키를 입력하세요: ")
os.environ["OPENAI_API_KEY"] = api_key

# 기본 임포트
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings, HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chat_models import ChatOpenAI
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate

print("설정 완료!")

OpenAI API 키를 입력하세요: ··········
설정 완료!


In [ ]:
def upload_and_process_pdf():
    """
    사용자가 PDF 파일을 업로드하고 이를 처리하는 함수
    """
    print("PDF 파일을 업로드해주세요...")
    uploaded = files.upload()

    # 업로드된 파일이 없는 경우
    if not uploaded:
        print("파일이 업로드되지 않았습니다.")
        return None, None

    # 첫 번째 업로드된 파일 처리
    filename = list(uploaded.keys())[0]

    # 임시 파일로 저장
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix='.pdf')
    temp_file.write(uploaded[filename])
    temp_path = temp_file.name
    temp_file.close()

    print(f"'{filename}' 파일이 업로드되었습니다. 처리를 시작합니다...")

    # PDF 파일 로드
    loader = PyPDFLoader(temp_path)
    documents = loader.load()

    print(f"PDF에서 {len(documents)} 페이지를 로드했습니다.")

    # 텍스트 분할
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len
    )

    chunks = text_splitter.split_documents(documents)

    print(f"문서를 {len(chunks)} 개의 청크로 분할했습니다.")

    # 임시 파일 삭제
    os.unlink(temp_path)

    return chunks, filename

# PDF 업로드 및 처리 실행
document_chunks, pdf_filename = upload_and_process_pdf()

if document_chunks is None:
    print("PDF 처리에 실패했습니다. 다시 시도해주세요.")


PDF 파일을 업로드해주세요...


Saving 키오스크(무인정보단말기) 이용실태 조사.pdf to 키오스크(무인정보단말기) 이용실태 조사.pdf
'키오스크(무인정보단말기) 이용실태 조사.pdf' 파일이 업로드되었습니다. 처리를 시작합니다...
PDF에서 59 페이지를 로드했습니다.
문서를 110 개의 청크로 분할했습니다.


In [ ]:
def create_vector_store(chunks):
    """
    문서 청크로부터 벡터 스토어를 생성하는 함수
    """
    if chunks is None or len(chunks) == 0:
        print("처리할 문서 청크가 없습니다.")
        return None

    print("임베딩 모델을 초기화하고 벡터 스토어를 생성합니다...")

    # 두 가지 임베딩 옵션 제공
    print("임베딩 모델을 선택하세요:")
    print("1. OpenAI 임베딩 (더 정확하지만 API 비용 발생)")
    print("2. HuggingFace 임베딩 (무료지만 정확도가 상대적으로 낮을 수 있음)")

    choice = input("선택 (1 또는 2): ")

    if choice == "1":
        embeddings = OpenAIEmbeddings()
        print("OpenAI 임베딩 모델을 사용합니다.")
    else:
        # mpnet은 다국어를 지원하는 훌륭한 임베딩 모델입니다
        embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
        )
        print("HuggingFace 임베딩 모델을 사용합니다.")

    # FAISS 벡터 스토어 생성
    try:
        vector_store = FAISS.from_documents(chunks, embeddings)
        print("벡터 스토어 생성이 완료되었습니다!")
        return vector_store
    except Exception as e:
        print(f"벡터 스토어 생성 중 오류가 발생했습니다: {e}")
        return None

# 벡터 스토어 생성
vector_store = create_vector_store(document_chunks) if document_chunks else None

if vector_store is None:
    print("벡터 스토어 생성에 실패했습니다. 다시 시도해주세요.")


임베딩 모델을 초기화하고 벡터 스토어를 생성합니다...
임베딩 모델을 선택하세요:
1. OpenAI 임베딩 (더 정확하지만 API 비용 발생)
2. HuggingFace 임베딩 (무료지만 정확도가 상대적으로 낮을 수 있음)
선택 (1 또는 2): 2


<ipython-input-7-d1d7a3ce18a1>:23: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.90k [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

HuggingFace 임베딩 모델을 사용합니다.
벡터 스토어 생성이 완료되었습니다!


In [ ]:
def setup_rag_chain(vector_store):
    """
    검색 증강 생성(RAG) 체인을 설정하는 함수
    """
    if vector_store is None:
        print("벡터 스토어가 없어 RAG 체인을 설정할 수 없습니다.")
        return None

    print("RAG 체인을 구성합니다...")

    # 모델 선택
    print("사용할 모델을 선택하세요:")
    print("1. gpt-3.5-turbo (빠르고 경제적)")
    print("2. gpt-4 (더 정확하지만 더 느리고 비쌈)")

    model_choice = input("선택 (1 또는 2): ")

    # 선택에 따른 모델 설정
    if model_choice == "2":
        llm = ChatOpenAI(temperature=0, model="gpt-4")
        print("GPT-4 모델을 사용합니다.")
    else:
        llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")
        print("GPT-3.5-Turbo 모델을 사용합니다.")

    # 대화 메모리 설정
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True
    )

    # 검색기 설정
    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 3}  # 상위 3개의 관련 청크 검색
    )

    # 프롬프트 템플릿 설정
    qa_template = """
    당신은 PDF 문서의 내용에 기반하여 질문에 답변하는 도우미입니다.

    주어진 정보만을 사용하여 질문에 답변하세요. 정보가 충분하지 않다면, "주어진 문서에서 해당 정보를 찾을 수 없습니다"라고 답변하세요.

    항상 문서의 내용에 충실하게 답변하고, 추측하지 마세요.

    질문: {question}

    관련 문서 내용:
    {context}

    답변:
    """

    QA_PROMPT = PromptTemplate(
        template=qa_template,
        input_variables=["question", "context"]
    )

    # RAG 체인 구성
    qa_chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=retriever,
        memory=memory,
        combine_docs_chain_kwargs={"prompt": QA_PROMPT}
    )

    print("RAG 체인 구성이 완료되었습니다!")
    return qa_chain

# RAG 체인 설정
qa_chain = setup_rag_chain(vector_store) if vector_store else None

if qa_chain is None:
    print("RAG 체인 설정에 실패했습니다. 다시 시도해주세요.")


RAG 체인을 구성합니다...
사용할 모델을 선택하세요:
1. gpt-3.5-turbo (빠르고 경제적)
2. gpt-4 (더 정확하지만 더 느리고 비쌈)
선택 (1 또는 2): 2


<ipython-input-8-21ed1f815d6c>:20: LangChainDeprecationWarning: The class `ChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import ChatOpenAI``.
  llm = ChatOpenAI(temperature=0, model="gpt-4")


GPT-4 모델을 사용합니다.
RAG 체인 구성이 완료되었습니다!


<ipython-input-8-21ed1f815d6c>:27: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(


In [ ]:
def chat_with_pdf(qa_chain, pdf_filename):
    """
    PDF 문서에 대한 대화형 인터페이스를 제공하는 함수
    """
    if qa_chain is None:
        print("RAG 체인이 설정되지 않아 대화를 시작할 수 없습니다.")
        return

    print(f"\n===== {pdf_filename} 문서와의 대화를 시작합니다 =====")
    print("질문을 입력하면 PDF 문서의 내용을 기반으로 답변해 드립니다.")
    print("대화를 종료하려면 '종료'를 입력하세요.\n")

    while True:
        question = input("질문: ")

        if question.lower() in ['종료', 'exit', 'quit']:
            print("\n대화를 종료합니다. 감사합니다!")
            break

        if question.strip() == "":
            print("질문을 입력해주세요.")
            continue

        try:
            # 질문을 처리하고 응답 생성
            print("\n답변을 생성 중입니다...")
            response = qa_chain({"question": question})

            print("\n답변:")
            print(response["answer"])
            print("\n" + "-" * 50)

        except Exception as e:
            print(f"\n오류가 발생했습니다: {e}")
            print("다시 시도해주세요.\n")

# 대화형 인터페이스 실행
if qa_chain and pdf_filename:
    chat_with_pdf(qa_chain, pdf_filename)
else:
    print("PDF 문서와 대화를 시작하기 위한 설정이 완료되지 않았습니다.")



===== 키오스크(무인정보단말기) 이용실태 조사.pdf 문서와의 대화를 시작합니다 =====
질문을 입력하면 PDF 문서의 내용을 기반으로 답변해 드립니다.
대화를 종료하려면 '종료'를 입력하세요.

질문: 무인정보 단말기 실태

답변을 생성 중입니다...


<ipython-input-9-6f10ae66a999>:27: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = qa_chain({"question": question})



답변:
무인정보 단말기인 키오스크의 이용 실태에 대해 조사한 결과, 모든 연령대에서 '뒷사람 눈치가 보이고', '조작이 어렵고', '상품 검색이 어려운 점'에 불편을 느꼈습니다. 특히 20대부터 50대까지는 '주문 첫 화면으로 돌아가는 법을 몰라서'나 '원하는 상품이 없어서' 이용을 중단한 경우가 많았으며, 60대 이상에서는 '뒷사람 눈치가 보여서' 이용을 중단한 사례가 가장 많았습니다. 또한, 매장에 직원이 있는데도 키오스크로만 주문할 수 있다고 안내받은 사례가 많았고, '외식업'에서는 '주문 실수를 인지하지 못해 다른 상품을 받은 사례'가 많았습니다. 이용 만족도는 60대 이상에서 가장 낮았고, 40대에서 가장 높았습니다. 전체 500명 중 233명(46.6%)이 최근 1년간 키오스크 이용 중 불편하거나 피해를 경험했다고 응답했습니다. 이용이 불편한 이유 중 가장 많은 응답은 '주문이 늦어지면 뒷사람 눈치가 보인다(52.8%)'였고, 그 다음으로 '메뉴 조작이 어려워서'46.8%, '기기가 잘 작동하지 않아서'39.1%였습니다.

--------------------------------------------------
질문: 키오스크 관련 사회적 이슈

답변을 생성 중입니다...

답변:
주어진 문서에서 해당 정보를 찾을 수 없습니다.

--------------------------------------------------
질문: 대표적 디지털 약자층

답변을 생성 중입니다...

답변:
장애인과 고령자가 대표적인 디지털 약자층입니다.

--------------------------------------------------


KeyboardInterrupt: Interrupted by user

In [ ]:
def visualize_conversation_history(qa_chain):
    """
    현재까지의 대화 이력을 시각화하는 함수
    """
    if qa_chain is None or not hasattr(qa_chain, 'memory') or qa_chain.memory is None:
        print("대화 이력을 가져올 수 없습니다.")
        return

    try:
        # 대화 이력 가져오기
        chat_history = qa_chain.memory.chat_memory.messages

        if not chat_history:
            print("아직 대화 이력이 없습니다.")
            return

        print("\n===== 대화 이력 =====")

        for i, message in enumerate(chat_history):
            # 메시지 타입에 따라 다르게 표시
            if hasattr(message, 'type') and message.type == 'human':
                print(f"질문 {i//2 + 1}: {message.content}")
            elif hasattr(message, 'type') and message.type == 'ai':
                print(f"답변 {i//2 + 1}: {message.content[:100]}..." if len(message.content) > 100 else f"답변 {i//2 + 1}: {message.content}")
                print("-" * 50)

    except Exception as e:
        print(f"대화 이력 시각화 중 오류가 발생했습니다: {e}")

# 이 함수를 실행하려면 다음과 같이 호출하세요:
visualize_conversation_history(qa_chain)



===== 대화 이력 =====
질문 1: 무인정보 단말기 실태
답변 1: 무인정보 단말기인 키오스크의 이용 실태에 대해 조사한 결과, 모든 연령대에서 '뒷사람 눈치가 보이고', '조작이 어렵고', '상품 검색이 어려운 점'에 불편을 느꼈습니다. 특히 2...
--------------------------------------------------
질문 2: 키오스크 관련 사회적 이슈
답변 2: 주어진 문서에서 해당 정보를 찾을 수 없습니다.
--------------------------------------------------
질문 3: 대표적 디지털 약자층
답변 3: 장애인과 고령자가 대표적인 디지털 약자층입니다.
--------------------------------------------------


In [ ]:
def process_multiple_pdfs():
    """
    여러 PDF 파일을 업로드하고 처리하는 함수
    """
    print("여러 PDF 파일을 업로드해주세요...")
    uploaded = files.upload()

    # 업로드된 파일이 없는 경우
    if not uploaded:
        print("파일이 업로드되지 않았습니다.")
        return None

    all_chunks = []
    filenames = []

    # 각 업로드된 파일 처리
    for filename in uploaded.keys():
        if not filename.lower().endswith('.pdf'):
            print(f"'{filename}'은(는) PDF 파일이 아닙니다. 건너뜁니다.")
            continue

        # 임시 파일로 저장
        temp_file = tempfile.NamedTemporaryFile(delete=False, suffix='.pdf')
        temp_file.write(uploaded[filename])
        temp_path = temp_file.name
        temp_file.close()

        print(f"'{filename}' 파일을 처리합니다...")

        try:
            # PDF 파일 로드
            loader = PyPDFLoader(temp_path)
            documents = loader.load()

            print(f"- {len(documents)} 페이지를 로드했습니다.")

            # 텍스트 분할
            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=1000,
                chunk_overlap=200,
                length_function=len
            )

            chunks = text_splitter.split_documents(documents)

            # 각 청크의 메타데이터에 파일명 추가
            for chunk in chunks:
                if 'source' not in chunk.metadata:
                    chunk.metadata['source'] = filename

            print(f"- {len(chunks)} 개의 청크로 분할했습니다.")

            all_chunks.extend(chunks)
            filenames.append(filename)

            # 임시 파일 삭제
            os.unlink(temp_path)

        except Exception as e:
            print(f"'{filename}' 처리 중 오류가 발생했습니다: {e}")
            continue

    if not all_chunks:
        print("처리된 PDF 문서가 없습니다.")
        return None

    print(f"총 {len(all_chunks)} 개의 청크를 {len(filenames)} 개의 PDF 파일에서 추출했습니다.")

    # 벡터 스토어 생성
    print("\n모든 문서에 대한 벡터 스토어를 생성합니다...")
    try:
        # OpenAI 임베딩 사용
        embeddings = OpenAIEmbeddings()
        vector_store = FAISS.from_documents(all_chunks, embeddings)
        print("벡터 스토어 생성이 완료되었습니다!")
        return vector_store, ", ".join(filenames)

    except Exception as e:
        print(f"벡터 스토어 생성 중 오류가 발생했습니다: {e}")
        return None

# 이 기능을 사용하려면 주석을 해제하고 실행하세요:
multi_pdf_vector_store, multi_pdf_filenames = process_multiple_pdfs()
multi_pdf_qa_chain = setup_rag_chain(multi_pdf_vector_store) if multi_pdf_vector_store else None
if multi_pdf_qa_chain and multi_pdf_filenames:
    chat_with_pdf(multi_pdf_qa_chain, f"여러 PDF 문서({multi_pdf_filenames})")


여러 PDF 파일을 업로드해주세요...


KeyboardInterrupt: 

In [ ]:
def setup_rag_chain_with_source_tracking(vector_store):
    """
    소스 추적 기능이 있는 검색 증강 생성(RAG) 체인을 설정하는 함수
    """
    if vector_store is None:
        print("벡터 스토어가 없어 RAG 체인을 설정할 수 없습니다.")
        return None

    print("소스 추적 기능이 있는 RAG 체인을 구성합니다...")

    # 모델 초기화
    llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")

    # 대화 메모리 설정
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True
    )

    # 검색기 설정 - 더 많은 문서 검색
    retriever = vector_store.as_retriever(
        search_type="similarity",
        search_kwargs={"k": 5}  # 상위 5개의 관련 청크 검색
    )

    # 프롬프트 템플릿 설정 - 소스 인용 요청 추가
    qa_template = """
    당신은 PDF 문서의 내용에 기반하여 질문에 답변하는 도우미입니다.

    주어진 정보만을 사용하여 질문에 답변하세요. 정보가 충분하지 않다면, "주어진 문서에서 해당 정보를 찾을 수 없습니다"라고 답변하세요.

    항상 문서의 내용에 충실하게 답변하고, 추측하지 마세요.

    반드시 답변의 끝에 정보의 출처를 [출처: 파일명, 페이지번호] 형식으로 명시하세요.
    여러 출처를 사용한 경우 모두 나열하세요.

    질문: {question}

    관련 문서 내용:
    {context}

    답변:
    """

    QA_PROMPT = PromptTemplate(
        template=qa_template,
        input_variables=["question", "context"]
    )

    # RAG 체인 구성 - 소스 반환 옵션 추가
    qa_chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=retriever,
        memory=memory,
        combine_docs_chain_kwargs={"prompt": QA_PROMPT},
        return_source_documents=True  # 소스 문서 반환 옵션 활성화
    )

    print("소스 추적 기능이 있는 RAG 체인 구성이 완료되었습니다!")
    return qa_chain

def chat_with_pdf_and_sources(qa_chain, pdf_filename):
    """
    소스 추적 기능이 있는 PDF 문서와의 대화형 인터페이스
    """
    if qa_chain is None:
        print("RAG 체인이 설정되지 않아 대화를 시작할 수 없습니다.")
        return

    print(f"\n===== {pdf_filename} 문서와의 대화를 시작합니다 (소스 추적 기능 포함) =====")
    print("질문을 입력하면 PDF 문서의 내용을 기반으로 답변해 드리고, 정보의 출처를 표시합니다.")
    print("대화를 종료하려면 '종료'를 입력하세요.\n")

    while True:
        question = input("질문: ")

        if question.lower() in ['종료', 'exit', 'quit']:
            print("\n대화를 종료합니다. 감사합니다!")
            break

        if question.strip() == "":
            print("질문을 입력해주세요.")
            continue

        try:
            # 질문을 처리하고 응답 생성
            print("\n답변을 생성 중입니다...")
            response = qa_chain({"question": question})

            # 응답 출력
            print("\n답변:")
            print(response["answer"])

            # 소스 문서 정보 추출 및 표시
            if "source_documents" in response:
                print("\n참고한 문서 정보:")
                sources_seen = set()  # 중복 제거를 위한 집합

                for i, doc in enumerate(response["source_documents"]):
                    source_info = f"- "

                    # 파일명 정보 추출
                    if 'source' in doc.metadata:
                        source_info += f"파일: {doc.metadata['source']}"

                    # 페이지 정보 추출
                    if 'page' in doc.metadata:
                        source_info += f", 페이지: {doc.metadata['page']}"

                    # 중복되지 않은 경우에만 출력
                    if source_info not in sources_seen:
                        sources_seen.add(source_info)
                        print(source_info)

            print("\n" + "-" * 50)

        except Exception as e:
            print(f"\n오류가 발생했습니다: {e}")
            print("다시 시도해주세요.\n")

# 소스 추적 기능을 사용하려면 주석을 해제하고 실행하세요:
source_tracking_qa_chain = setup_rag_chain_with_source_tracking(vector_store)
if source_tracking_qa_chain and pdf_filename:
    chat_with_pdf_and_sources(source_tracking_qa_chain, pdf_filename)


소스 추적 기능이 있는 RAG 체인을 구성합니다...
소스 추적 기능이 있는 RAG 체인 구성이 완료되었습니다!

===== 키오스크(무인정보단말기) 이용실태 조사.pdf 문서와의 대화를 시작합니다 (소스 추적 기능 포함) =====
질문을 입력하면 PDF 문서의 내용을 기반으로 답변해 드리고, 정보의 출처를 표시합니다.
대화를 종료하려면 '종료'를 입력하세요.



KeyboardInterrupt: Interrupted by user